# Stock-correlation networks on real returns: Phase A notebook

**Status: Phase A. Methodology, code and tests are complete; the empirical run is gated on real price data that is not yet in this repository.** When the required files are absent the notebook states what is missing and stops. It never substitutes simulated market paths, and no figure or number below is produced without real data.

Question: do networks built from rolling correlations carry stable, out-of-sample information about diversification beyond classical covariance tools?

## Decisions (details in `METHODOLOGY.md`)

| Item | Decision |
|---|---|
| Universe | Point-in-time S&P 500 membership; a stock enters a window only if it was a member on the formation date and has at least 95% valid returns in the lookback |
| Prices | Daily split- and dividend-adjusted closes (total-return convention); returns are simple close-to-close; missing values are never filled |
| Survivorship | Measured, not assumed away: `coverage_by_date` compares point-in-time members with available price series; the backtest refuses to run through a held name that loses data |
| Correlation window | 252 trading days (sensitivity: 126 and 504) |
| Distance | Mantegna metric d = sqrt(2(1 - rho)) |
| Network | Minimum spanning tree (n - 1 edges); metrics: total length, degree concentration, leaf share, first-eigenvalue share, mean path length |
| Communities | Greedy modularity on the MST, compared with sector labels by adjusted Rand index and modularity |
| Stability | Edge Jaccard between windows; community persistence (adjusted Rand index) |
| Evaluation | Walk-forward, monthly rebalance at the close, weights from data up to that close, holding returns strictly after it |
| Baselines | Equal weight, long-only minimum variance (Ledoit-Wolf shrinkage), Hierarchical Risk Parity; plus an exploratory MST inverse-degree rule |
| Frictions | One-way proportional costs on traded notional (0, 10, 25 bps), turnover reported |

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = (Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
from stocknet import data as dt, networks as nw, portfolios as pf, backtest as bt, metrics as mt, viz

DATA = ROOT / "data"
PRICES = DATA / "prices_adjusted_close.csv"
MEMBERSHIP = DATA / "sp_500_historical_components.csv"
SECTORS = DATA / "sectors.csv"                      # columns: ticker, sector (point-in-time sector labels are preferable; see METHODOLOGY.md)
LOOKBACK, COSTS_BPS, START = 252, (0, 10, 25), "2005-01-01"
EXPORT, FIG_DIR = True, ROOT / "results" / "figures"

required = {"adjusted close prices": PRICES, "point-in-time membership": MEMBERSHIP}
missing = [name for name, p in required.items() if not p.exists()]
DATA_AVAILABLE = not missing
print("Required inputs:", {k: p.exists() for k, p in required.items()}, "| sectors file:", SECTORS.exists())
if missing:
    print("\nDATA NOT AVAILABLE:", ", ".join(missing))
    print("No analysis is run and no results are produced. See data/README.md for the exact files and how to create them.")

Required inputs: {'adjusted close prices': False, 'point-in-time membership': False} | sectors file: False

DATA NOT AVAILABLE: adjusted close prices, point-in-time membership
No analysis is run and no results are produced. See data/README.md for the exact files and how to create them.


## 1. Data validation and survivorship check

In [2]:
if DATA_AVAILABLE:
    prices = dt.load_prices(PRICES)
    membership = dt.load_membership(MEMBERSHIP, start="2003-01-01")
    prov_path = PRICES.with_suffix(".provenance.json")
    print(json.dumps(json.loads(prov_path.read_text()), indent=2) if prov_path.exists() else "WARNING: no provenance file next to the prices")
    q = dt.quality_report(prices)
    print(f"{q.n_tickers} tickers, {q.n_dates} dates; tickers with short history: {len(q.short_history)}")
    print(f"suspect jumps (|r| > 40%): {len(q.suspect_jumps)}, of which reversed next day (likely data errors): {int(q.suspect_jumps.reversed.sum()) if len(q.suspect_jumps) else 0}")
    display(q.suspect_jumps.sort_values("ret").head(15))
    cov = dt.coverage_by_date(membership, prices, pd.date_range("2005-12-30", prices.index[-1], freq="YE").map(lambda d: prices.index[prices.index.searchsorted(d) - 1]))
    display(cov.style.format({"coverage": "{:.1%}"}))
    print("Coverage below 100% means point-in-time members without a usable price series (mostly delisted or renamed names): results are conditional on survivors.")

## 2. Rolling correlation, distance and MST structure

In [3]:
if DATA_AVAILABLE:
    returns = dt.simple_returns(prices)
    rebal = bt.month_end_dates(returns.index, start=START)
    uni = lambda d: dt.eligible_universe(membership, returns, d, LOOKBACK)
    rows, graphs = [], {}
    for d, c, g in nw.rolling_networks(returns, rebal, LOOKBACK, uni):
        rows.append(dict(date=d, **nw.network_metrics(g, c)))
        graphs[d] = g
    metrics_df = pd.DataFrame(rows).set_index("date")
    display(metrics_df.describe().T.round(4))

## 3. Communities versus sectors, and stability over time

In [4]:
if DATA_AVAILABLE and SECTORS.exists():
    sectors = pd.read_csv(SECTORS).set_index("ticker").sector.to_dict()
    ari, mod_sector, mod_comm, comm_prev = [], [], [], None
    comm_by_date = {}
    for d, g in graphs.items():
        comm = nw.communities(g)
        labels = {n: hash(sectors.get(n, "unknown")) % 10**6 for n in g.nodes}
        ari.append(dict(date=d, ari_vs_sector=nw.adjusted_rand_index(comm, labels)))
        mod_sector.append(nw.modularity_of_labels(g, labels)); mod_comm.append(nw.modularity_of_labels(g, comm))
        comm_by_date[d] = comm
    sec_df = pd.DataFrame(ari).set_index("date").assign(modularity_sector=mod_sector, modularity_communities=mod_comm)
    display(sec_df.describe().T.round(3))
elif DATA_AVAILABLE:
    print("sectors.csv not provided: community-versus-sector comparison skipped.")

if DATA_AVAILABLE:
    dates = list(graphs)
    sim = pd.DataFrame(index=dates, columns=dates, dtype=float)
    for i, a in enumerate(dates):
        for b in dates[i:]:
            sim.loc[a, b] = sim.loc[b, a] = nw.edge_jaccard(graphs[a], graphs[b])
    lag1 = pd.Series([nw.edge_jaccard(graphs[a], graphs[b]) for a, b in zip(dates[:-1], dates[1:])], index=dates[1:])
    print(f"Month-to-month MST edge persistence (Jaccard): median {lag1.median():.2f}, 5th percentile {lag1.quantile(.05):.2f}")

## 4. Walk-forward portfolio evaluation
Weights at each month-end use only returns up to and including that day; holding returns are strictly later. Costs are charged on traded notional at the rebalance.

In [5]:
if DATA_AVAILABLE:
    rules = pf.RULES
    results = {}
    for cost in COSTS_BPS:
        for name, rule in rules.items():
            results[(name, cost)] = bt.run_backtest(returns, membership, rule, rebal, lookback=LOOKBACK, cost_bps=cost)
    summary = pd.DataFrame({k: mt.summarize(v.net, v.turnover) for k, v in results.items()}).T
    summary.index.names = ["strategy", "cost_bps"]
    display(summary.round(4))
    net10 = pd.DataFrame({n: results[(n, 10)].net for n in rules})
    print("Sharpe difference vs equal weight at 10 bps (moving-block bootstrap on observed strategy returns):")
    for n in rules:
        if n != "equal_weight":
            print(n, mt.block_bootstrap_diff(net10[n], net10["equal_weight"]))

## 5. Candidate figures (written only when real results exist)

In [6]:
if DATA_AVAILABLE and EXPORT:
    last = dates[-1]
    groups = {n: sectors.get(n, "unknown") for n in graphs[last].nodes} if SECTORS.exists() else None
    viz.export(viz.fig_mst(graphs[last], groups), FIG_DIR, "mst-latest", (1100, 700))
    viz.export(viz.fig_metric_lines(metrics_df, {"mst_mean_edge": "Mean MST edge distance", "first_eigen_share": "First-eigenvalue share", "max_degree": "Largest MST degree"}), FIG_DIR, "rolling-structure", (1100, 760))
    viz.export(viz.fig_stability_heatmap(sim), FIG_DIR, "edge-persistence", (1000, 700))
    viz.export(viz.fig_equity_curves(net10), FIG_DIR, "equity-curves-net-10bps", (1100, 560))
    print("Figures written to", FIG_DIR)
elif not DATA_AVAILABLE:
    print("No figures: real data is not available.")

No figures: real data is not available.


## Limitations to carry into Phase B
* A price source without delisted names gives a survivor universe; the coverage table quantifies the bias but does not remove it. Strategy comparisons within one biased universe are only relatively informative.
* Sector labels are usually current rather than point-in-time.
* The backtest ignores shorting, taxes, market impact and delisting returns (it refuses to run through missing held names instead).
* Overlapping windows are strongly dependent; confidence intervals use block resampling of observed strategy returns and are descriptive.
* Network rules are exploratory hypotheses, not recommendations. Nothing here is investment advice.